# ALS validation tuning

> **Results status (2026-11-22):** The previously saved grid outputs have been cleared. The old run `2026-11-16T01-34-52.248706Z` is a historical conditional-ranking diagnostic only; its scores and conclusions are not results for the current catalog-wide ranking objective. No corrected 16-configuration grid is currently reported. Rerun all 16 configurations before drawing tuning conclusions; a single checked configuration is not a completed grid. The current CSV exporter includes `conditionalPrecisionAt10`, `conditionalNdcgAt10`, and `rankingCandidatePairs`.

Compare a 16-configuration ALS grid using a validation split made only from the deterministic outer training set. For each fit, measure validation RMSE plus Precision@10 and NDCG@10 against validation movies rated at least 4.0. The ranking metrics rank the same candidates as the baseline and calibration pipeline: every learned catalog movie the user did not rate in the fit partition. The held-out test partition is never used by this experiment.

The grid holds `maxIter=10`, ALS seed `42`, and `coldStartStrategy="drop"` fixed while varying rank and regularization. The proposal's baseline (`rank=50`, `regParam=0.1`) remains unchanged in `als-baseline.ipynb`; this notebook only reports how it compares. Spark runs with `local[2]` and the project's approximately 1 GB driver heap. The compact MovieLens 1M factors and per-fit cleanup keep the search bounded; if a configuration fails due to memory, record the failed configuration and available heap rather than increasing concurrency or silently omitting it.


## Start Spark and verify the runtime

Use Spark 3.5.9, Scala 2.12.18, Java 17, and two local worker threads. The synthetic count verifies that Spark can execute before reading MovieLens data.


In [ ]:
import $ivy.`org.apache.spark::spark-sql:3.5.9`
import $ivy.`org.apache.spark::spark-mllib:3.5.9`

import org.apache.spark.sql.SparkSession


In [ ]:
val spark = SparkSession.builder()
  .appName("CS 7265 - ALS Validation Tuning")
  .master("local[2]")
  .config("spark.driver.host", "127.0.0.1")
  .config("spark.driver.bindAddress", "127.0.0.1")
  .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
val scalaVersion = scala.util.Properties.versionNumberString
val javaVersion = System.getProperty("java.version")
val environmentCheck = spark.range(3).count()
assert(spark.version == "3.5.9")
assert(scalaVersion == "2.12.18")
assert(System.getProperty("java.specification.version") == "17")
assert(environmentCheck == 3L)
println(s"Spark: ${spark.version}; Scala: $scalaVersion; Java: $javaVersion")
println(s"PASS: environment check on $environmentCheck synthetic rows")


## Load ratings and construct only the outer training partition

Use the canonical per-user split from `popularity-profiles.ipynb`: order by `xxhash64(42, userId, movieId)` then `movieId`, and assign `floor(0.8 * userRatingCount)` rows to training. This cell creates only `trainingRatings`; it does not construct, count, cache, or otherwise consume test rows. The validation split below is derived solely from `trainingRatings`.


In [ ]:
case class Rating(userId: Int, movieId: Int, rating: Float, timestamp: Long)

class RatingParser extends (String => Rating) with Serializable {
  def apply(line: String): Rating = {
    val fields = line.split("::")
    require(fields.length == 4, "Expected four fields per MovieLens rating")
    Rating(fields(0).toInt, fields(1).toInt, fields(2).toFloat, fields(3).toLong)
  }
}


In [ ]:
val ratingsPath = Seq(
  "data/ml-1m/ratings.dat",
  "../data/ml-1m/ratings.dat"
).find(path => new java.io.File(path).isFile).getOrElse {
  throw new IllegalArgumentException("Place MovieLens 1M ratings.dat in data/ml-1m/ and run from the repository root or notebooks folder.")
}
val repositoryRoot = new java.io.File(ratingsPath).getCanonicalFile
  .getParentFile.getParentFile.getParentFile
val ratingsRDD = spark.sparkContext.textFile(ratingsPath).map(new RatingParser)
import spark.implicits._
val ratingsDF = ratingsRDD.toDF().cache()
val ratingCount = ratingsDF.count()
require(ratingCount == 1000209L, s"Expected 1000209 MovieLens ratings, found $ratingCount")
println(s"PASS: loaded $ratingCount ratings")


In [ ]:
import org.apache.spark.sql.expressions.Window
import org.apache.spark.sql.functions._

val trainingFraction = 0.8
val splitSeed = 42L
val perUser = Window.partitionBy("userId")
val ratingsWithCounts = ratingsDF.withColumn("userRatingCount", count("*").over(perUser))
val ratingsWithKeys = ratingsWithCounts.withColumn(
  "shuffleKey", xxhash64(lit(splitSeed), col("userId"), col("movieId")))
val shuffledPerUser = Window.partitionBy("userId")
  .orderBy(col("shuffleKey"), col("movieId"))
val rankedRatings = ratingsWithKeys.withColumn("ratingPosition", row_number().over(shuffledPerUser))
val ratingsWithCutoff = rankedRatings.withColumn(
  "trainingCount", floor(col("userRatingCount") * trainingFraction))
val trainingRatings = ratingsWithCutoff
  .filter(col("ratingPosition") <= col("trainingCount"))
  .select("userId", "movieId", "rating", "timestamp")
  .persist(org.apache.spark.storage.StorageLevel.MEMORY_AND_DISK)
val trainingRatingCount = trainingRatings.count()
assert(trainingRatingCount == 797758L,
  s"Expected 797758 outer-training ratings, found $trainingRatingCount")
println(s"PASS: outer training partition contains $trainingRatingCount ratings; test rows are not evaluated")


## Create the validation split from trainingRatings only

Within the outer training partition, independently order each user's rows by `xxhash64(7, userId, movieId)` then `movieId`; use the first `floor(0.8 * n)` rows to fit and the rest for validation. Confirm the partitions are nonempty and disjoint, and that their counts add back to the outer training count.


In [ ]:
import org.apache.spark.sql.DataFrame

// Per-user hash split; the input is explicitly the outer training partition.
def perUserSplit(df: DataFrame, seed: Long, fraction: Double): (DataFrame, DataFrame) = {
  require(fraction > 0.0 && fraction < 1.0, "Split fraction must be strictly between 0 and 1.")
  val ranked = df
    .withColumn("n", count("*").over(Window.partitionBy("userId")))
    .withColumn("k", xxhash64(lit(seed), col("userId"), col("movieId")))
    .withColumn("pos", row_number().over(
      Window.partitionBy("userId").orderBy(col("k"), col("movieId"))))
    .withColumn("cut", floor(col("n") * fraction))
  val fit = ranked.filter(col("pos") <= col("cut"))
    .select("userId", "movieId", "rating", "timestamp")
  val validation = ranked.filter(col("pos") > col("cut"))
    .select("userId", "movieId", "rating", "timestamp")
  (fit, validation)
}

val (trainFit, validation) = perUserSplit(trainingRatings, seed = 7L, fraction = 0.8)
trainFit.persist(org.apache.spark.storage.StorageLevel.MEMORY_AND_DISK)
validation.persist(org.apache.spark.storage.StorageLevel.MEMORY_AND_DISK)
val trainFitCount = trainFit.count()
val validationCount = validation.count()
val validationOverlap = trainFit.select("userId", "movieId")
  .join(validation.select("userId", "movieId"), Seq("userId", "movieId"), "inner")
  .limit(1).count()
assert(trainFitCount > 0L && validationCount > 0L,
  s"Validation partitions must be nonempty: fit=$trainFitCount, validation=$validationCount")
assert(trainFitCount + validationCount == trainingRatingCount,
  s"Validation split lost rows: fit=$trainFitCount + validation=$validationCount, outer=$trainingRatingCount")
assert(validationOverlap == 0L, "Per-user fit and validation partitions overlap.")
println(s"PASS: seed=7 training-only split trainFit=$trainFitCount, validation=$validationCount, overlap=$validationOverlap")
ratingsDF.unpersist(blocking = true)
trainingRatings.unpersist(blocking = true)


## Run the grid with bounded per-fit memory

The only model input is `trainFit`. RMSE is evaluated over validation ratings that ALS can score (`coldStartStrategy="drop"`). For Precision@10 and NDCG@10, each eligible user (at least one validation rating `>= 4.0`) is scored against every learned catalog movie they did not rate in `trainFit`, the same candidate universe the baseline and calibration notebooks rank. Predictions are ranked by score (movie ID breaks ties), and the top 10 is compared with that user's validation movies rated `>= 4.0`. The notebook checks that every candidate pair is scored and that every eligible user receives exactly 10 movies. Liked validation movies that are absent from `trainFit` cannot be recommended and count as misses, as in the pipeline.

`conditionalPrecisionAt10` and `conditionalNdcgAt10` are a conditional diagnostic only: they rank just the movies each user rated in validation, which makes them far higher than the pipeline's top-N metrics. They are reported for comparison with earlier runs and are not used to judge tuning.

Each model is fit sequentially. Materialized predictions and model factor tables are unpersisted in `finally`, references are released, and garbage collection is requested before the next fit. `fitSeconds` reports model-fitting time; `evaluationSeconds` reports prediction and metric time. Results are saved in a new timestamped run folder using create-new semantics.


In [ ]:
import org.apache.spark.ml.evaluation.RegressionEvaluator
import org.apache.spark.ml.recommendation.ALS
import org.apache.spark.mllib.evaluation.RankingMetrics
import org.apache.spark.sql.functions._

case class TuningResult(
  rank: Int,
  regParam: Double,
  maxIter: Int,
  seed: Long,
  rmse: Double,
  precisionAt10: Double,
  ndcgAt10: Double,
  conditionalPrecisionAt10: Double,
  conditionalNdcgAt10: Double,
  scoredValidationRatings: Long,
  rankingCandidatePairs: Long,
  eligibleUsers: Long,
  fitSeconds: Double,
  evaluationSeconds: Double
)

val ranks = Seq(10, 20, 50, 100)
val regParams = Seq(0.01, 0.05, 0.1, 0.2)
val maxIter = 10
val alsSeed = 42L
val listSize = 10
val validationRelevant = validation.filter(col("rating") >= 4.0)
  .select("userId", "movieId").cache()
val relevantCount = validationRelevant.count()
val eligibleUsers = validationRelevant.select("userId").distinct().cache()
val eligibleUserCount = eligibleUsers.count()
require(relevantCount > 0L && eligibleUserCount > 0L,
  s"Expected relevant validation items; relevant=$relevantCount, users=$eligibleUserCount")
println(s"PASS: $relevantCount validation items rated >= 4 from $eligibleUserCount eligible users")

// Ranking candidates mirror the baseline/calibration pipeline: for each eligible user, every learned
// catalog movie that user did not rate in trainFit. Their trainFit pairs are excluded from candidates.
val eligibleFitPairs = trainFit.select("userId", "movieId")
  .join(eligibleUsers, Seq("userId"), "left_semi")
  .persist(org.apache.spark.storage.StorageLevel.MEMORY_AND_DISK)
val eligibleFitPairCount = eligibleFitPairs.count()
val trainFitMovieCount = trainFit.select("movieId").distinct().count()
val expectedRankingCandidatePairs = eligibleUserCount * trainFitMovieCount - eligibleFitPairCount
println(s"PASS: ranking candidates = $eligibleUserCount users x $trainFitMovieCount catalog movies - " +
  s"$eligibleFitPairCount fit pairs = $expectedRankingCandidatePairs")

val actualLists = validationRelevant.groupBy("userId")
  .agg(collect_set("movieId").as("actual")).cache()
def rankingMetricsFor(predictedLists: DataFrame): (Double, Double) = {
  // Eligible users without a predicted list stay in the evaluation with an empty list.
  val rankingRows = actualLists.join(eligibleUsers, Seq("userId"), "inner")
    .join(predictedLists, Seq("userId"), "left")
    .select(coalesce(col("predicted"), array().cast("array<int>")).as("predicted"), col("actual"))
  val rankingRdd = rankingRows.rdd.map { row =>
    (row.getSeq[Int](0).toArray, row.getSeq[Int](1).toArray)
  }
  val metrics = new RankingMetrics(rankingRdd)
  (metrics.precisionAt(listSize), metrics.ndcgAt(listSize))
}

val rmseEvaluator = new RegressionEvaluator()
  .setMetricName("rmse").setLabelCol("rating").setPredictionCol("prediction")
val resultRows = scala.collection.mutable.ArrayBuffer.empty[TuningResult]

def evaluateConfiguration(rank: Int, regParam: Double): TuningResult = {
  val als = new ALS()
    .setUserCol("userId").setItemCol("movieId").setRatingCol("rating")
    .setImplicitPrefs(false)
    .setRank(rank).setMaxIter(maxIter).setRegParam(regParam).setSeed(alsSeed)
    .setColdStartStrategy("drop")

  val fitStarted = System.nanoTime()
  val model = als.fit(trainFit)
  val fitSeconds = (System.nanoTime() - fitStarted).toDouble / 1e9
  var scoredValidation: DataFrame = null
  var rankingCandidates: DataFrame = null
  try {
    val evaluationStarted = System.nanoTime()
    scoredValidation = model.transform(validation)
      .filter(col("prediction").isNotNull && !isnan(col("prediction")))
      .persist(org.apache.spark.storage.StorageLevel.MEMORY_AND_DISK)
    val scoredValidationRatings = scoredValidation.count()
    require(scoredValidationRatings > 0L,
      s"No validation ratings scored for rank=$rank, regParam=$regParam")
    val rmse = rmseEvaluator.evaluate(scoredValidation)

    // Top-N objective: rank every learned catalog movie the user did not rate in trainFit, as the
    // baseline/calibration pipeline does, by prediction (movie ID breaks ties).
    val catalogMovies = model.itemFactors.select(col("id").as("movieId"))
    val catalogMovieCount = catalogMovies.count()
    require(catalogMovieCount == trainFitMovieCount,
      s"ALS learned $catalogMovieCount movies, expected the $trainFitMovieCount movies in trainFit")
    rankingCandidates = model.transform(eligibleUsers.crossJoin(catalogMovies)
        .join(eligibleFitPairs, Seq("userId", "movieId"), "left_anti"))
      .persist(org.apache.spark.storage.StorageLevel.DISK_ONLY)
    val rankingCandidatePairs = rankingCandidates.count()
    val invalidCandidatePredictions = rankingCandidates
      .filter(col("prediction").isNull || isnan(col("prediction"))).count()
    require(rankingCandidatePairs == expectedRankingCandidatePairs,
      s"Expected $expectedRankingCandidatePairs ranking candidates, scored $rankingCandidatePairs")
    require(invalidCandidatePredictions == 0L, "Every ranking candidate must receive a finite prediction.")
    val candidateOrder = Window.partitionBy("userId").orderBy(col("prediction").desc, col("movieId").asc)
    val predictedLists = rankingCandidates
      .withColumn("candidateRank", row_number().over(candidateOrder))
      .filter(col("candidateRank") <= listSize)
      .groupBy("userId")
      .agg(sort_array(collect_list(struct(col("candidateRank"), col("movieId")))).as("ranked"))
      .select(col("userId"), expr("transform(ranked, x -> x.movieId)").as("predicted"))
      .cache()
    require(predictedLists.count() == eligibleUserCount &&
      predictedLists.filter(size(col("predicted")) =!= listSize).count() == 0L,
      s"Every eligible user must receive exactly $listSize ranked catalog candidates.")
    val (precisionAt10, ndcgAt10) = rankingMetricsFor(predictedLists)
    predictedLists.unpersist(blocking = true)

    // Conditional diagnostic only: ranks just the movies each user rated in validation. Its values are
    // not comparable to the pipeline's top-N metrics and are not used to judge tuning.
    val conditionalLists = scoredValidation.groupBy("userId")
      .agg(slice(sort_array(collect_list(struct(
        (-col("prediction")).as("negativePrediction"), col("movieId").as("movieId")))),
        1, listSize).as("ranked"))
      .select(col("userId"), expr("transform(ranked, x -> x.movieId)").as("predicted"))
    val (conditionalPrecisionAt10, conditionalNdcgAt10) = rankingMetricsFor(conditionalLists)
    val evaluationSeconds = (System.nanoTime() - evaluationStarted).toDouble / 1e9
    require(Seq(rmse, precisionAt10, ndcgAt10, conditionalPrecisionAt10, conditionalNdcgAt10)
      .forall(x => !x.isNaN && !x.isInfinity),
      s"Non-finite metric for rank=$rank, regParam=$regParam")
    TuningResult(rank, regParam, maxIter, alsSeed, rmse,
      precisionAt10, ndcgAt10, conditionalPrecisionAt10, conditionalNdcgAt10,
      scoredValidationRatings, rankingCandidatePairs, eligibleUserCount,
      fitSeconds, evaluationSeconds)
  } finally {
    if (scoredValidation != null) scoredValidation.unpersist(blocking = true)
    if (rankingCandidates != null) rankingCandidates.unpersist(blocking = true)
    model.userFactors.unpersist(blocking = true)
    model.itemFactors.unpersist(blocking = true)
    System.gc()
  }
}

for (rank <- ranks; regParam <- regParams) {
  val result = evaluateConfiguration(rank, regParam)
  resultRows += result
  println(f"PASS: rank=$rank%3d regParam=$regParam%.2f fit=${result.fitSeconds}%.2fs " +
    f"eval=${result.evaluationSeconds}%.2fs RMSE=${result.rmse}%.6f " +
    f"P@10=${result.precisionAt10}%.6f NDCG@10=${result.ndcgAt10}%.6f " +
    f"scored=${result.scoredValidationRatings}%d")
}
assert(resultRows.size == ranks.size * regParams.size,
  s"Expected ${ranks.size * regParams.size} grid results, found ${resultRows.size}")
println(s"PASS: completed ${resultRows.size} distinct ALS configurations")


## Compare settings and save results

Display every grid row sorted by validation RMSE and by validation NDCG@10 (catalog-wide ranking). “Optimal across all objectives” means the proposal's `(rank=50, regParam=0.1)` ties for the best RMSE, Precision@10, and NDCG@10 simultaneously; because these objectives may disagree, the notebook also states its standing for each measure separately. This report does not change the ALS baseline configuration.


In [ ]:
val results = resultRows.toSeq.toDF().cache()
assert(results.count() == 16L, "Expected exactly 16 grid rows.")
assert(results.select("rank", "regParam").distinct().count() == 16L,
  "Grid contains duplicate or missing rank/regParam configurations.")

println("Validation results sorted by RMSE (ascending):")
results.orderBy(col("rmse").asc, col("rank").asc, col("regParam").asc)
  .show(20, truncate = false)
println("Validation results sorted by NDCG@10 (descending):")
results.orderBy(col("ndcgAt10").desc, col("rank").asc, col("regParam").asc)
  .show(20, truncate = false)

val resultByConfig = resultRows.map(r => (r.rank, r.regParam) -> r).toMap
val proposal = resultByConfig((50, 0.1))
val bestRmse = resultRows.map(_.rmse).min
val bestPrecision = resultRows.map(_.precisionAt10).max
val bestNdcg = resultRows.map(_.ndcgAt10).max
val metricTieTolerance = 1e-12
val proposalBestRmse = math.abs(proposal.rmse - bestRmse) <= metricTieTolerance
val proposalBestPrecision = math.abs(proposal.precisionAt10 - bestPrecision) <= metricTieTolerance
val proposalBestNdcg = math.abs(proposal.ndcgAt10 - bestNdcg) <= metricTieTolerance
val optimalOnAll = proposalBestRmse && proposalBestPrecision && proposalBestNdcg
println(s"Proposal rank=50/regParam=0.1: RMSE=${proposal.rmse}, P@10=${proposal.precisionAt10}, NDCG@10=${proposal.ndcgAt10}")
println(s"Best grid values: RMSE=$bestRmse; P@10=$bestPrecision; NDCG@10=$bestNdcg")
println(s"Proposal standing: RMSE ${if (proposalBestRmse) "best (tied)" else "not best"}; " +
  s"P@10 ${if (proposalBestPrecision) "best (tied)" else "not best"}; " +
  s"NDCG@10 ${if (proposalBestNdcg) "best (tied)" else "not best"}.")
println(if (optimalOnAll)
  "FINDING: rank 50 / regParam 0.1 is optimal across all three measured validation objectives."
else
  "FINDING: rank 50 / regParam 0.1 is not uniformly optimal across RMSE, Precision@10, and NDCG@10; see the sorted tables for the trade-offs.")

val tuningRunId = java.time.Instant.now().toString.replace(":", "-")
val tuningOutputDir = new java.io.File(repositoryRoot, s"outputs/als-tuning/$tuningRunId")
require(tuningOutputDir.mkdirs(),
  s"Could not create unique tuning output folder (it may already exist): ${tuningOutputDir.getPath}")
val gridPath = new java.io.File(tuningOutputDir, "grid.csv").toPath
val writer = java.nio.file.Files.newBufferedWriter(gridPath,
  java.nio.charset.StandardCharsets.UTF_8,
  java.nio.file.StandardOpenOption.CREATE_NEW,
  java.nio.file.StandardOpenOption.WRITE)
try {
  writer.write("rank,regParam,maxIter,seed,rmse,precisionAt10,ndcgAt10,conditionalPrecisionAt10,conditionalNdcgAt10,scoredValidationRatings,rankingCandidatePairs,eligibleUsers,fitSeconds,evaluationSeconds")
  writer.newLine()
  resultRows.foreach { r =>
    writer.write(Seq(r.rank.toString, r.regParam.toString, r.maxIter.toString,
      r.seed.toString, r.rmse.toString, r.precisionAt10.toString, r.ndcgAt10.toString,
      r.conditionalPrecisionAt10.toString, r.conditionalNdcgAt10.toString,
      r.scoredValidationRatings.toString, r.rankingCandidatePairs.toString, r.eligibleUsers.toString,
      r.fitSeconds.toString, r.evaluationSeconds.toString).mkString(","))
    writer.newLine()
  }
} finally {
  writer.close()
}
assert(java.nio.file.Files.size(gridPath) > 0L, "Grid CSV was not written.")
println(s"PASS: saved ${resultRows.size} grid results to ${gridPath.toString}")


## Stop Spark

Unpersist the shared validation data and stop the local session after saving the results.


In [ ]:
validationRelevant.unpersist(blocking = true)
actualLists.unpersist(blocking = true)
eligibleFitPairs.unpersist(blocking = true)
eligibleUsers.unpersist(blocking = true)
trainFit.unpersist(blocking = true)
validation.unpersist(blocking = true)
results.unpersist(blocking = true)
spark.stop()
println("PASS: cached validation data released and Spark session stopped")
